# Tecan Infinite M1000 Pro

```{device-card} tecan-infinite-m1000-pro
```

| Property | Value |
| --- | --- |
| Connection | USB |
| Experimental operations | Plate carrier, temperature, 600 nm absorbance |

**Experimental:** tray motion was reported successful after initialization, but an earlier opening attempt produced abnormal stuttering. Stop immediately if movement is abnormal. Temperature control, absorbance acquisition and the updated sequence handling have only been tested offline; they are not hardware-verified.

**Do not operate the tray:** opening has stuttered after a power cycle. Initialization and safe operation are under investigation; a reported position does not prove that the tray moved correctly.

## Connection and safety

Connect the reader over USB using a driver supported by PyLabRobot. On Windows, installing a libusbK driver may prevent i-control from communicating until its Tecan driver is restored. Let the reader complete its normal startup; ensure the carrier path is clear. Obtain appropriate approval before each hardware operation. After an error or unexpected movement, stop, inspect the instrument and resolve its state before reconnecting.

## Setup

`setup()` connects, initializes internal axes and replays the 89-command i-control startup exchange. **Experimental: this revised sequence has not been tested on hardware.** It changes instrument modes and may move internal mechanisms. Check that the instrument is clear, obtain explicit approval for each run, and supervise it. A failed or interrupted setup leaves instrument state uncertain.

In [ ]:
from pylabrobot.tecan import InfiniteM1000Pro

reader = InfiniteM1000Pro()
await reader.setup()

## Initialization compatibility

`setup()` already performs initialization. The following call is idempotent in the same successful session; it is not a recovery procedure. After an error or interruption, stop and assess the instrument before reconnecting.

In [ ]:
await reader.initialize()

## Open

Ensure the carrier path is clear before moving it out.

In [ ]:
await reader.open()

## Close

Place a compatible plate on the carrier if needed; keep the carrier path clear.

In [ ]:
await reader.close()

## Reader status

Query the instrument-reported status. This does not establish whether mechanisms have physically stopped.

In [ ]:
status = await reader.request_instrument_status()
status

## Read absorbance

The current acquisition supports **only** the captured, uncovered physical GRE96ft 96-well plate, 600 nm and all wells. Place that physical plate in the carrier, close it, and supervise the read. The resource below is used only as a logical set of well names; it is not a declaration that a Bio-Rad plate matches the instrument's fixed coordinates. Do not use a different physical plate.

Omit `wells` or pass `None` for the full plate. `wells=[]` and partial selections are rejected. The sequence has not yet been tested with the revised startup/sequence handling. Results are not automatically blank corrected.

In [ ]:
from pylabrobot.resources.biorad.plates import biorad_96_wellplate_200uL_Vb

well_map = biorad_96_wellplate_200uL_Vb("well_map_only")
result = await reader.read_absorbance(well_map, wavelength=600, plate_definition="GRE96ft")
result.data

## Read fluorescence — experimental reference only

**Hardware-unverified.** This fixed method is based on one i-control capture and simulated transport tests. It moves internal mechanisms. Before running, confirm the physical plate is an **uncovered GRE96ft 96-well plate**, the carrier is IN, and the instrument is clear; obtain per-run approval and supervise the operation. Stop if there is abnormal sound or movement. The `well_map` created above supplies logical well names only; it does not indicate that a Bio-Rad plate is physically compatible.

The method reads all 96 wells with top optics, 230 nm excitation, 280 nm emission, 5 nm excitation/emission bandwidth, gain 100, and 50 flashes. Arbitrary wavelengths, plates, and subsets are not supported. Treat the returned values as experimental, not independently validated quantitative measurements.

In [ ]:
fluorescence = await reader.read_fluorescence_reference(
  well_map, plate_definition="GRE96ft"
)
fluorescence.data

## Read temperature

This operation is experimental. Use only with an appropriately supervised, approved session.

In [ ]:
temperature = await reader.request_current_temperature()
temperature

## Enable temperature control

**Not hardware-verified.** A target in the range 20–40 °C is accepted. This changes the reader state; use only with explicit approval and appropriate supervision.

In [ ]:
await reader.set_temperature(30.0)

## Disable temperature control

Stop heating or temperature regulation when it is no longer needed.

In [ ]:
await reader.stop_temperature_control()

## Disconnect

Disconnecting does not move the carrier.

In [ ]:
await reader.stop()